# Week 6 — Feature Engineering, Scaling, Encoding and Skewness
**Netflix Contents Patterns Analysis**  
**Sahana Viyani A. | 25BAD092 | AI & DS**  
**Kumaraguru College of Technology**

## Aim
Prepare Netflix catalog data for numerical analysis by creating useful features, encoding categories, scaling numerical columns, and examining skewness. The dataset describes **titles**, not individual viewing habits.

## 1. Import libraries and load the raw CSV

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style="whitegrid")
# Put netflix_titles.csv beside this notebook, or update DATA_PATH.
DATA_PATH = Path("netflix_titles.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Place the original netflix_titles.csv in the same folder as this notebook.")
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("Missing values:")
print(df.isna().sum())

## 2. Feature engineering
Extract years and months, a primary genre, primary country, duration units, and counts of genres/countries.

In [ ]:
# Feature engineering from Netflix catalog metadata (not viewer watch histories)
data = df.copy()
for col in ['type','rating','listed_in','country','duration']:
    if col not in data.columns: data[col] = np.nan
for col in ['release_year','date_added']:
    if col not in data.columns: data[col] = np.nan
# Dates sometimes have extra spaces; invalid dates become missing.
data['date_added_clean'] = pd.to_datetime(data['date_added'].astype('string').str.strip(), errors='coerce')
data['year_added'] = data['date_added_clean'].dt.year
data['month_added'] = data['date_added_clean'].dt.month
# Extract numeric duration and separate units so minutes and seasons aren't compared as equivalent.
data['duration_value'] = pd.to_numeric(data['duration'].astype('string').str.extract(r'(\d+)', expand=False), errors='coerce')
data['duration_unit'] = np.select([data['duration'].astype('string').str.contains('min', case=False, na=False), data['duration'].astype('string').str.contains('season', case=False, na=False)], ['Minutes','Seasons'], default='Unknown')
data['genre_primary'] = data['listed_in'].fillna('Unknown').astype(str).str.split(',').str[0].str.strip()
data['country_primary'] = data['country'].fillna('Unknown').astype(str).str.split(',').str[0].str.strip()
data['genre_count'] = data['listed_in'].fillna('').astype(str).apply(lambda x: len([v for v in x.split(',') if v.strip()]))
data['country_count'] = data['country'].fillna('').astype(str).apply(lambda x: len([v for v in x.split(',') if v.strip()]))
data['release_year'] = pd.to_numeric(data['release_year'], errors='coerce')
data['content_age_at_addition'] = (data['year_added'] - data['release_year']).clip(lower=0)
print('Engineered features:')
display(data[['type','genre_primary','duration_value','duration_unit','genre_count','country_count','year_added','content_age_at_addition']].head())

## 3. Select usable columns and handle missing values
Use median imputation for numerical values and the most frequent category for categorical values.

In [ ]:
from sklearn.impute import SimpleImputer
numeric_cols = ['release_year','year_added','month_added','duration_value','genre_count','country_count','content_age_at_addition']
category_cols = ['type','rating','genre_primary','duration_unit']
num_imp = SimpleImputer(strategy='median')
cat_imp = SimpleImputer(strategy='most_frequent')
X_num = pd.DataFrame(num_imp.fit_transform(data[numeric_cols]), columns=numeric_cols, index=data.index)
X_cat = pd.DataFrame(cat_imp.fit_transform(data[category_cols]), columns=category_cols, index=data.index)
print('Missing after imputation:', int(X_num.isna().sum().sum()+X_cat.isna().sum().sum()))
display(X_num.head())

## 4. Normalization and standardization
MinMaxScaler maps values to 0–1; StandardScaler centers and scales features.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
minmax = pd.DataFrame(MinMaxScaler().fit_transform(X_num), columns=X_num.columns)
standard = pd.DataFrame(StandardScaler().fit_transform(X_num), columns=X_num.columns)
display(minmax.head())
print('Standardized means (approximately zero):')
print(standard.mean().round(3))

## 5. Encode categorical variables
One-hot encoding creates numeric columns for each category. Limit genre to the 10 most frequent groups to keep the table manageable.

In [ ]:
top_genres = X_cat['genre_primary'].value_counts().head(10).index
X_cat_small = X_cat.copy()
X_cat_small['genre_primary'] = X_cat_small['genre_primary'].where(X_cat_small['genre_primary'].isin(top_genres), 'Other')
encoded = pd.get_dummies(X_cat_small, dtype=int)
print('Encoded shape:', encoded.shape)
display(encoded.head())

## 6. Check skewness and correct it
Use `log1p` for non-negative skewed features; do not apply it blindly to year fields.

In [ ]:
skew_cols = ['duration_value','genre_count','country_count','content_age_at_addition']
before = X_num[skew_cols].skew()
corrected = X_num[skew_cols].copy()
for col in skew_cols:
    if before[col] > 0.75 and (corrected[col] >= 0).all():
        corrected[col] = np.log1p(corrected[col])
after = corrected.skew()
display(pd.DataFrame({'Before':before,'After':after}).round(3))
fig, axes = plt.subplots(1,2,figsize=(12,4))
sns.histplot(X_num['duration_value'], bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Duration before transformation')
sns.histplot(corrected['duration_value'], bins=30, ax=axes[1], color='darkorange')
axes[1].set_title('Duration after transformation (if skewed)')
plt.tight_layout(); plt.show()

## 7. Compare scales visually

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4))
sns.boxplot(data=minmax[['duration_value','genre_count','country_count']], ax=axes[0]); axes[0].set_title('Min-max normalized')
sns.boxplot(data=standard[['duration_value','genre_count','country_count']], ax=axes[1]); axes[1].set_title('Standardized')
plt.tight_layout(); plt.show()

## 8. Save prepared features for Week 7

In [ ]:
# Correct skewed columns first, then standardize all numerical features.
final_numeric = X_num.copy()
final_numeric[skew_cols] = corrected
final_scaled = pd.DataFrame(StandardScaler().fit_transform(final_numeric), columns=final_numeric.columns, index=data.index)
week6_features = pd.concat([final_scaled, encoded.set_axis(data.index)], axis=1)
week6_features.to_csv('netflix_week6_features.csv', index=False)
print('Saved netflix_week6_features.csv:', week6_features.shape)
display(week6_features.head())

## Observation / conclusion
Fill in the actual numbers after running the notebook. Explain which columns were most skewed, which transformations were applied, and how many features were created. Scaling and encoding prepare catalog attributes for PCA in Week 7.